In [ ]:

from fastapi import APIRouter, Depends, HTTPException

from app.api.auth import get_current_user
from app.services.recommendation_service import RecommendationService


router = APIRouter(
    prefix="/recommendations",
    tags=["Recommendations"],
)


def get_database():
    from app.main import app

    database = getattr(app.state, "database", None)

    if database is None:
        raise RuntimeError("Database is not initialized.")

    return database


@router.get("")
async def get_recommendations(
    current_user=Depends(get_current_user),
):
    database = get_database()

    recommendations = list(
        database.collection("recommendations").find(
            {
                "user_id": current_user.id,
                "status": "pending",
            },
            {
                "_id": 0,
            },
        ).sort(
            "created_at",
            -1,
        )
    )

    project_ids = {
        item.get("project_id")
        for item in recommendations
        if item.get("project_id")
    }

    projects = {}

    if project_ids:
        project_list = database.collection("projects").find(
            {
                "user_id": current_user.id,
                "id": {
                    "$in": list(project_ids),
                },
            },
            {
                "_id": 0,
                "id": 1,
                "name": 1,
            },
        )

        for project in project_list:
            projects[project["id"]] = project["name"]

    for item in recommendations:
        item["project_name"] = projects.get(
            item.get("project_id")
        )

    return recommendations


@router.post("/generate/{project_id}")
async def generate_recommendations(
    project_id: str,
    current_user=Depends(get_current_user),
):
    database = get_database()

    project = database.collection("projects").find_one(
        {
            "id": project_id,
            "user_id": current_user.id,
        }
    )

    if project is None:
        raise HTTPException(
            status_code=404,
            detail="Project not found.",
        )

    service = RecommendationService(
        database=database
    )

    return service.generate_for_project(
        user_id=current_user.id,
        project_id=project_id,
        goals=(
            [project["goal"]]
            if project.get("goal")
            else []
        ),
    )
